# XCP-D mean time series: a beginner-friendly workflow

This notebook is the continuation of the XCP-D tutorial.

Our goal is to take one XCP-D mean time series file and turn it into something useful for network analysis.

We will do the following:

1. Load a real XCP-D mean time series file
2. Look at the shape of the data
3. Compute a parcel-by-parcel correlation matrix
4. Flatten the matrix to get pairwise correlations
5. Convert Pearson r values to Fisher z values
6. Collapse parcels into broader networks
7. Do a simple seed-based analysis

This is exactly the kind of workflow you will use when moving from preprocessing to connectivity analysis.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns 

# This points to the XCP-D derivatives we are using for this tutorial.
# You can replace this with a path frpm your or project folder.
XCP_DIR = Path('/projects/aabdulrasul/Tutorials/derivatives/xcp_d')

# get subjects from the derivatives folder and sort them
subjects = sorted([sub.name for sub in XCP_DIR.glob('sub-*')])
print(f'Found {len(subjects)} subjects: {subjects}')




In [ ]:

# XCP has a tonne of output files, we can use the glob function to find the mean time series files for each subject. The mean time series files are named according to the following pattern: 

# organize the mean time series files by subject
mean_ts_files_by_subject = {}
for sub in subjects:
    mean_ts_files_by_subject[sub] = sorted(XCP_DIR.glob(f'{sub}/**/*-mean_timeseries.tsv')) # easier just to glob all the mean time series files and then organize them by atlas(seg) and parcel 


# number of mean time series files for each subject
num_mean_ts_files = {sub: len(files) for sub, files in mean_ts_files_by_subject.items()}
print(f'Number of mean time series files for each subject: {num_mean_ts_files}')

We can organize in a nested dictionary where the keys are the subject, task, run, atlas and parcel and the values are the file paths to the mean time series files. This will make it easier to access the mean time series files for a specific subject, task, run, atlas and parcel.

In [ ]:
# organize the mean time series per subject by task, run, atlas

# we can use string operations to extract this info based on the bids file naming convention

# e.g sub-<participant_id>_<session_id>_<task>_<run>_<space>_<atlas>_<stat>-mean_timeseries.tsv
#                0               1        2      3      4       5      6

# if we split the file name by '_' we get 6 pieces, the task is the 3rd piece, the run is the 4th piece, and the atlas is the 6th piece. remember we count from 0


mean_ts_files_by_subject_and_atlas = {}

for sub, files in mean_ts_files_by_subject.items(): # this for loop over subjects, and their respective files in the dictionary mean_ts_files_by_subject.
    mean_ts_files_by_subject_and_atlas[sub] = {} # we initialize an empty dictionary for each subject in the new dictionary mean_ts_files_by_subject_and_atlas.
    for file in files: # for all the files in the list of files for EACH subject, we extract the file name into the parts and select the parts we want to organize by
        # extract the task, run, atlas from the file name
        task = file.name.split('_')[2] 
        run = file.name.split('_')[3]
        atlas = file.name.split('_')[5] 
        
        if task not in mean_ts_files_by_subject_and_atlas[sub]: # we initialize an empty dictionary 
            mean_ts_files_by_subject_and_atlas[sub][task] = {} # for each task in the dictionary to collect all the mean time series for each task
        if run not in mean_ts_files_by_subject_and_atlas[sub][task]: # same thing for run
            mean_ts_files_by_subject_and_atlas[sub][task][run] = {} # catches all the runs for each task in the dictionary to collect all the mean time series for each run
            
        # Assign the file directly to the atlas key. now we make the dictionary structure as follows: mean_ts_files_by_subject_and_atlas[sub][task][run][atlas] = file
        # so we can call on the mean time series file for a specific subject, task, run, and atlas by using the dictionary keys in that order.
        mean_ts_files_by_subject_and_atlas[sub][task][run][atlas] = file


## 1. Load the XCP-D mean time series

A mean time series file has rows as time points and columns as parcels.

For example, if you're using Schafer 156, 156 columns, and the numbers of rows will be how many TRs in the scan


This means each parcel has its own time series across the scan.

For the purpose of this tutorial, we will focus on just one subject.

We will use their resting state scans, in the schafer 156 space.

In [ ]:
# pull the first subject, resting state, all runs, and in the schafer456 space
sub = subjects[0]
task = 'task-rest'
atlas = 'seg-4S156Parcels'

# load both task runs mean time series files into pandas dataframes
mean_ts_run_01 = pd.read_csv(mean_ts_files_by_subject_and_atlas[sub][task]['run-01'][atlas], sep='\t')
mean_ts_run_02 = pd.read_csv(mean_ts_files_by_subject_and_atlas[sub][task]['run-02'][atlas], sep='\t')

print(f"Shape of mean time series for run 01: {mean_ts_run_01.shape}")
print(f"Shape of mean time series for run 02: {mean_ts_run_02.shape}")

# display the first few rows of the mean time series for run 01
mean_ts_run_01.head()

Now that we have our timeseries loaded for one specific person, lets concatenate them and run a correlation matrix

In [ ]:
# stack the two runs together into a single dataframe
mean_ts_concatenated = pd.concat([mean_ts_run_01, mean_ts_run_02], axis=0) # using pandas, concat the two dataframes along the rows (axis=0, if you used 1 we would be concatenating along the columns)

mean_ts_concatenated.shape # check the shape of the new dataframe, it should have the same number of columns as the original dataframes, but the number of rows should be the sum of the two original dataframes


## 2. Compute a parcel-by-parcel correlation matrix

Now we ask: how similar is parcel A to parcel B across time?

This gives us a connectivity matrix where each value is a Pearson correlation coefficient.

A value close to 1 means strong positive coupling.
A value close to 0 means weak or no linear relationship.
A value close to -1 means strong negative coupling.

In [ ]:
# Compute the parcel-by-parcel correlation matrix.
# The result is a 456 x 456 matrix.
correlation_matrix = mean_ts_concatenated.corr(method='pearson')

print('Correlation matrix shape:', correlation_matrix.shape)


In [ ]:
## We can also plot the correlation matrix as a heatmap. 

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, cmap='coolwarm', center=0)
plt.title('Pearson R Correlation Matrix of the mean time series')
plt.show()

## 3. Preprocessing Correlation Matrix

- Fisher (Z) transformation
- 0 the diagonal (self correlations)
- Extract upper triangle 

In [ ]:
# Fisher transformation of the correlation matrix.
# This is often done to stabilize the variance of correlation coefficients. Also makes interpretation a lot easier. 
# zero out the self correlations (diagonal) to avoid infinities in the Fisher transformation.
# Z transformation is very common in the field, but as always, consult with literature, supervisor and as a sanity check, run a correlation between both matrices to see if they are highly correlated.

# using numpy arctanh function to perform the fisher transformation on the correlation matrix. The arctanh function is the inverse hyperbolic tangent function, which is used to perform the Fisher transformation. 
z_matrix = np.arctanh(correlation_matrix.values) 

# print the z matrix first 5 rows and columns
print(z_matrix[:5, :5])

# notice how there are some infinities in the z matrix, this is because the arctanh of 1 is undefined. therefore the self correlations (diagonal) are set to 0 

np.fill_diagonal(z_matrix, 0) # set the diagonal to 0 

print(z_matrix[:5, :5]) # print the z matrix first 5 rows and columns after setting the diagonal to 0



In [ ]:
# we can now plot the fisher transformation. lets first conver the z_matrix back to a pandas dataframe

z_matrix_df = pd.DataFrame(z_matrix, index=correlation_matrix.index, columns=correlation_matrix.columns)

## plot the correlation matrix as a heatmap. 

plt.figure(figsize=(10, 8))
sns.heatmap(z_matrix_df, cmap='coolwarm', center=0)
plt.title(' Fisher Z-Transformed Correlation Matrix of the mean time series')
plt.show()

In [ ]:
# sanity check, we can compute the correlation between the original correlation matrix and the z matrix to see if they are highly correlated.
# we can use the numpy corrcoef function to compute the correlation between the two matrices. The corrcoef function computes the Pearson correlation coefficient between two matrices. We can use the flatten method to convert the matrices to 1D arrays before computing the correlation.
correlation_between_matrices = np.corrcoef(correlation_matrix.values.flatten(), z_matrix.flatten())[0, 1]

correlation_between_matrices

#### Flatten the matrix

A full matrix is useful, but for many analyses we want a long table of pairwise correlations.

We can:

- keep only the upper triangle
- drop the diagonal (self-correlation)
- convert each r value to a Fisher z score using $z = \\text{atanh}(r)$

This step helps make the distribution of correlations more normal for downstream statistics.

In [ ]:
# Get the row/column positions for everything above the diagonal (k=1 excludes it)
rows, cols = np.triu_indices_from(z_matrix_df, k=1) # np.triu_indices_from returns the indices for the upper-triangle of an array. k=1 excludes the diagonal. so    
                                                    # so this gives us the row and column indices for all the elements in the upper triangle of the z_matrix_df. 
                                                    # we can use these indices to extract the values from the z_matrix_df and create a new dataframe with the 
                                                    # parcel pairs and their corresponding z values.

flat_corr = pd.DataFrame({ # now we can make a dataframe, using the row and column indices to get the parcel names, and the z values
    'parcel_1': z_matrix_df.index[rows], # so column 1 is the row index of the z_matrix_df, which is the parcel name for the first parcel in the pair
    'parcel_2': z_matrix_df.columns[cols], # so column 2 is the column index of the z_matrix_df, which is the parcel name for the second parcel in the pair
    'z': z_matrix_df.values[rows, cols] # tells us the corresponding z value for the parcel pair, using the row and column indices to get the value from the z_matrix_df
})

print('Number of pairwise correlations:', len(flat_corr))
flat_corr.head()

## 4. Network-level summary

The mean time series file contains parcels, not networks.

To answer questions like:

- How connected is the visual network to the default network?
- Are somatomotor parcels more connected with control parcels than with visual parcels?

we need to collapse parcels into larger groups.

A simple way to do this is to map parcel names to network labels using their names.
For example:

- LH_Vis_1 -> Visual
- RH_SomMot_2 -> SomMot
- LH_Default_7 -> Default

This is a beginner-friendly approximation, and in real analyses you should use the atlas/network labels that match your study design and documentation.

In [ ]:
# Define a small mapping from parcel-name prefixes to network names.
# This is intentionally simple and beginner-friendly. go through the columns yourself to see how the parcels are named and how they can be grouped into networks.
# note the last 56 columns are subcortical and cerebellar parcels, which we will ignore for this tutorial
# we are only using the 7 Yeo networks for this tutorial,
network_map = {
    'Vis': 'Visual',
    'SomMot': 'SomMot',
    'DorsAttn': 'DorsAttn',
    'SalVentAttn': 'SalVentAttn',
    'Limbic': 'Limbic',
    'Cont': 'Cont',
    'Default': 'Default'
}


# this function splits the parcel and we select the part that corresponds to the network name, 
# and then we use the network_map dictionary to convert the network name to a more readable format. 
# If the network name is not in the dictionary, we return 'Other'.

def parcel_to_network(parcel_name):  
    """Convert a parcel label like 'LH_Vis_1' to a network label."""
    parts = parcel_name.split('_')
    if len(parts) >= 2:
        prefix = parts[1]
        return network_map.get(prefix, 'Other')
    return 'Other'


# Label each parcel in the pair with its network 

flat_corr['network_1'] = flat_corr['parcel_1'].apply(parcel_to_network)
flat_corr['network_2'] = flat_corr['parcel_2'].apply(parcel_to_network)

# Drop pairs involving subcortical/cerebellar parcels, since we're only using the 7 Yeo networks
yeo_pairs = flat_corr[(flat_corr['network_1'] != 'Other') & (flat_corr['network_2'] != 'Other')].copy()

yeo_pairs.head()

## 5. Ask a question: how connected is network A to network B?

This is the kind of question you can answer once you have a network matrix.

For example:

- How connected are the visual and default networks?
- Is the somatomotor network more connected to dorsal attention than to limbic regions?
- Which network pairs show the strongest average coupling?

Now that we have our pairwise correlations and we identify what networks they come from. we can do some simple calculations to get the mean connectivity within network and between networks.


In [ ]:
# Put the two networks in a consistent order (alphabetical) so that
# Visual-Default and Default-Visual count as the same pair

network_low = np.minimum(yeo_pairs['network_1'], yeo_pairs['network_2'])
network_high = np.maximum(yeo_pairs['network_1'], yeo_pairs['network_2'])

yeo_pairs['network_a'] = network_low
yeo_pairs['network_b'] = network_high

# Average the z-values for each network pair
network_summary = yeo_pairs.groupby(['network_a', 'network_b'])['z'].mean().reset_index()


In [ ]:
# display the top 10 network pairs with the highest average z-values
network_summary.sort_values(by='z', ascending=False).head(10)

## 6. Seed-based analysis: pick one parcel and compare it to all others

A seed-based analysis asks:

- Which parcels are most strongly correlated with this seed parcel?

This is a simple and popular way to read a connectivity matrix.

Here we choose a visual parcel as the seed.

In [ ]:
seed_name = 'LH_Vis_1'

# Find every row where the seed shows up, on either side of the pair
seed_rows = flat_corr[ 
    (flat_corr['parcel_1'] == seed_name) | (flat_corr['parcel_2'] == seed_name)
].copy() # this uses pandas boolean indexing (if something is true, then select that row) to select all the rows in the flat_corr dataframe where either parcel_1 or parcel_2 is equal to the seed_name.
         # we use the copy() method to create a new dataframe that is a copy of the selected rows, so that we can modify it without affecting the original dataframe.

# the idea is that we want to find all the parcels that are correlated with the seed parcel, 
# and then we can sort them by their z-values to see which parcels are most strongly correlated with the seed parcel.

# flat corr dataframe has 3 columns: parcel_1, parcel_2, and z. we want to create a new dataframe that has 2 columns: other_parcel and z, where other_parcel 
# is the parcel that is correlated with the seed parcel, and z is the corresponding z-value.

# the seed though can be either parcel_1 or parcel_2, so we need to handle both cases. and not double count the same pair.

# we can do this by creating two separate dataframes, one for each case, and then concatenating them together.


# Now, for rows where the seed is parcel_1, so parcel_2 is the "other" parcel
case_1 = seed_rows[seed_rows['parcel_1'] == seed_name][['parcel_2', 'z']]
case_1 = case_1.rename(columns={'parcel_2': 'other_parcel'})

# Now, for rows where the seed is parcel_2, so parcel_1 is the "other" parcel
case_2 = seed_rows[seed_rows['parcel_2'] == seed_name][['parcel_1', 'z']]
case_2 = case_2.rename(columns={'parcel_1': 'other_parcel'})

# stack the two cases together into one table
seed_pairs = pd.concat([case_1, case_2])

# you now have a dataframe with that shows how the seed LH_Vis_1 is correlated with every other parcel in the brain, along with the corresponding z-values.
# seed based analysis is a common technique in fMRI analysis, where you select a seed region of interest and then look at how that region is functionally connected to other regions in the brain.
# for example, seeing how a component of the default mode is correlated with other regions in the brain. 
# Perhaps deviations from the normal pattern of connectivity are associated with a particular disease or disorder.



## 7. Save the outputs

Now that we have created the connectivity outputs, we can save them for later use.

This is helpful when you want to:

- inspect the matrix in another script
- visualize it in a separate notebook
- compare subjects or groups later

In [ ]:
# Save the outputs to a local folder so they can be reused.
out_dir = Path('/projects/aabdulrasul/Tutorials/fMRI_basics/outputs/xcpd_example')
out_dir.mkdir(parents=True, exist_ok=True)

correlation_matrix.to_csv(out_dir / 'parcel_correlation_matrix.tsv', sep='\t')
flat_corr.to_csv(out_dir / 'pairwise_correlations.tsv', sep='\t', index=False)
network_summary.to_csv(out_dir / 'network_correlation_matrix.tsv', sep='\t')
seed_pairs.to_csv(out_dir / 'seed_based_correlations.tsv', sep='\t', header=['correlation'])

print(f'Files written to {out_dir}')